# LLM Inference Benchmark Suite — Colab Orchestrator

Compares FP16, GPTQ, AWQ, GGUF, and TensorRT-LLM inference on a single
open-weight model, using **one isolated virtual environment per technique**
(see `README.md` and `docs/environment_notes.md` for why).

**Recommended runtime: A100 (40GB).** Phases 1-4 also work on a T4;
Phase 5 (TensorRT-LLM) requires Ampere or newer and will fail fast with a
clear assertion error otherwise.


In [ ]:
# Phase 0 -- Clone repo and confirm GPU
!git clone -q https://github.com/arkanathroy/llm-inference-benchmark-suite.git 2>/dev/null || echo "repo already present"
%cd llm-inference-benchmark-suite

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
# Phase 0.5 -- Shared Hugging Face cache (avoid re-downloading Qwen per env)
import os
from pathlib import Path

HF_CACHE_DIR = "/content/hf_cache"
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("No HF_TOKEN secret found -- proceeding unauthenticated (fine for public models like Qwen2.5-3B-Instruct).")

print(f"HF cache set to {HF_CACHE_DIR} -- shared across all envs/*/venv subprocesses "
      f"since HF_HOME is inherited via os.environ by every run_in_env() subprocess.")

In [ ]:
# Phase 0.6 -- Imports shared by every technique cell below
import sys, subprocess, time
sys.path.insert(0, "src")
from config import CONFIG
from env_runner import run_in_env
from server_utils import wait_for_health, stop_server, start_vllm_server, start_llamacpp_server
from pathlib import Path

def teardown_env(env_name):
    """Delete a technique's venv to reclaim disk space once its benchmark is done."""
    venv_path = Path(f"envs/{env_name}/venv")
    if venv_path.exists():
        get_ipython().system(f"rm -rf {venv_path}")
        print(f"Deleted {venv_path} to reclaim disk space.")
    get_ipython().system("df -h /content | tail -1")

def run_accuracy_eval(env_name, base_url, model_id, technique_name):
    """Run lm-eval accuracy tasks against a server that is already live,
    right after its throughput benchmark, avoiding a second server
    start/stop cycle. Results append to results/accuracy_results.csv.
    """
    eval_cfg = CONFIG.eval
    tasks_arg = ",".join(eval_cfg.tasks)
    print(f"[accuracy_eval] {technique_name}: running lm-eval tasks={tasks_arg} "
          f"limit={eval_cfg.limit} against {base_url} (this can take a few minutes)...")

    output_path = f"results/lm_eval_{technique_name}"
    run_in_env(
        env_name, "-m",
        ["lm_eval", "--model", "local-completions",
         "--model_args", f"base_url={base_url}/completions,model={model_id},num_concurrent=1,max_retries=3,tokenized_requests=False",
         "--tasks", tasks_arg,
         "--limit", str(eval_cfg.limit),
         "--output_path", output_path],
    )
    print(f"[accuracy_eval] {technique_name}: complete, results in {output_path}")

In [ ]:
# Phase 0.7 -- Start Loki + Promtail + Grafana for live log observability
# See docs/observability.md for the full stepwise guide (dashboards, panels,
# how to read logs live while a phase is running).
!bash observability/setup_observability.sh

# Expose Grafana (port 3000) via a public URL so you can view dashboards
# from your browser while Colab keeps running.
from google.colab.output import eval_js
grafana_url = eval_js("google.colab.kernel.proxyPort(3000)")
print(f"Grafana dashboard: {grafana_url}")
print("Open this URL, add a Loki data source at http://localhost:3100, "
      "then follow docs/observability.md to build the live progress panel.")

In [ ]:
# Phase A -- FP16 baseline: build env, benchmark, teardown
!bash envs/fp16/setup.sh

model_id = CONFIG.model.hf_repo
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"

server_proc = start_vllm_server(
    "envs/fp16/venv/bin/python", model_id, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len,
)
wait_for_health(base_url, server_proc)

run_in_env("fp16", "src/benchmark_runner.py",
           ["--technique", "fp16", "--base_url", f"{base_url}/v1", "--model_id", model_id])

run_accuracy_eval("fp16", f"{base_url}/v1", model_id, "fp16")

stop_server(server_proc)
teardown_env("fp16")

In [ ]:
# Phase B -- GPTQ: build env, quantize, benchmark, teardown
!bash envs/gptq/setup.sh

run_in_env("gptq", "src/quantize_gptq.py", [])

gptq_model_dir = CONFIG.gptq.output_dir
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"

server_proc = start_vllm_server(
    "envs/gptq/venv/bin/python", gptq_model_dir, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len, quantization="gptq",
)
wait_for_health(base_url, server_proc)

run_in_env("gptq", "src/benchmark_runner.py",
           ["--technique", "gptq", "--base_url", f"{base_url}/v1", "--model_id", gptq_model_dir])

run_accuracy_eval("gptq", f"{base_url}/v1", gptq_model_dir, "gptq")

stop_server(server_proc)
teardown_env("gptq")
# Optionally also free the quantized weights once benchmarked:
# get_ipython().system(f"rm -rf {gptq_model_dir}")

In [ ]:
# Phase C -- AWQ: build env, quantize, benchmark, teardown
!bash envs/awq/setup.sh

run_in_env("awq", "src/quantize_awq.py", [])

awq_model_dir = CONFIG.awq.output_dir
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"

server_proc = start_vllm_server(
    "envs/awq/venv/bin/python", awq_model_dir, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len, quantization="awq",
)
wait_for_health(base_url, server_proc)

run_in_env("awq", "src/benchmark_runner.py",
           ["--technique", "awq", "--base_url", f"{base_url}/v1", "--model_id", awq_model_dir])

run_accuracy_eval("awq", f"{base_url}/v1", awq_model_dir, "awq")

stop_server(server_proc)
teardown_env("awq")
# get_ipython().system(f"rm -rf {awq_model_dir}")

In [ ]:
# Phase D -- GGUF: build env, convert+quantize, benchmark, teardown
!bash envs/gguf/setup.sh

run_in_env("gguf", "src/convert_gguf.py", [])

gguf_dir = CONFIG.gguf.output_dir
gguf_file = f"{gguf_dir}/model-Q4_K_M.gguf"
llamacpp_port = CONFIG.server.llamacpp_port
llamacpp_base_url = f"http://{CONFIG.server.host}:{llamacpp_port}"

server_proc = start_llamacpp_server(
    "envs/gguf/llama.cpp/build/bin/llama-server", gguf_file,
    CONFIG.server.host, llamacpp_port, CONFIG.model.max_model_len,
)
wait_for_health(llamacpp_base_url, server_proc)

run_in_env("gguf", "src/benchmark_runner.py",
           ["--technique", "gguf_q4_k_m", "--base_url", f"{llamacpp_base_url}/v1", "--model_id", gguf_file])

run_accuracy_eval("gguf", f"{llamacpp_base_url}/v1", gguf_file, "gguf_q4_k_m")

stop_server(server_proc)
teardown_env("gguf")
# get_ipython().system(f"rm -rf {gguf_dir}")

In [ ]:
# Phase E -- TensorRT-LLM (Ampere+/A100 only): build env, build engine, benchmark, teardown
import torch
major, minor = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
if major >= 8:
    !bash envs/trtllm/setup.sh
    run_in_env("trtllm", "src/build_trtllm_engine.py", [])

    engine_dir = f"{CONFIG.trtllm.output_dir}/trt_engine"
    run_in_env("trtllm", "src/trtllm_bench.py",
               ["--engine_dir", engine_dir, "--output", "results/trtllm_benchmark.json"])

    teardown_env("trtllm")
    # get_ipython().system(f"rm -rf {CONFIG.trtllm.output_dir}")
else:
    print(f"Skipping TensorRT-LLM: GPU compute capability sm_{major}{minor} < sm_80 (needs Ampere+/A100).")

## Phase 9 -- Aggregate results and plot comparison charts

In [ ]:
# Phase 9.1 -- Load combined results CSV
import pandas as pd

results_csv = Path(CONFIG.output.results_dir) / CONFIG.output.csv_name
df = pd.read_csv(results_csv) if results_csv.exists() else pd.DataFrame()
df


In [ ]:
# Phase 9.2 -- Fold in TensorRT-LLM results (separate JSON schema) if present
import json

trtllm_json = Path("results/trtllm_benchmark.json")
if trtllm_json.exists():
    trt_results = json.load(open(trtllm_json))
    trt_df = pd.DataFrame(trt_results).rename(columns={
        "avg_latency_s": "avg_latency_s",
        "tokens_per_second": "avg_tokens_per_second",
    })
    df = pd.concat([df, trt_df], ignore_index=True, sort=False)

df


In [ ]:
# Phase 9.3 -- Plot tokens/sec by technique and batch size
import plotly.express as px

if not df.empty:
    fig = px.bar(
        df, x="batch_size", y="avg_tokens_per_second", color="technique",
        barmode="group", title="Throughput by technique and batch size",
        labels={"avg_tokens_per_second": "tokens / second", "batch_size": "batch size"},
    )
    fig.write_image("results/benchmark_comparison.png", scale=2)
    fig.show()
else:
    print("No results yet -- run Phases 3-7 first.")
